# EPED 2026 — Comunicação oral

**Título:** ANÁLISE EMPÍRICA DAS EXECUÇÕES FISCAIS ESTADUAIS PAULISTAS ENTRE 2016 E 2025  
**Evento:** XV Encontro Nacional de Pesquisa Empírica em Direito (EPED) · Belém/PA · 10–14 ago 2026  
**GT:** GT68 — Direito, tecnologia e inteligência artificial  
**ID submissão:** 1540866 · **Status:** Aprovado

Este notebook é o **entrypoint** da apresentação. A análise completa está em [`fesp_execucao_fiscal_analysis.ipynb`](fesp_execucao_fiscal_analysis.ipynb).

## 1. Objetivo

Descrever empiricamente o acervo de **execuções fiscais** no TJSP em que a **Fazenda do Estado de São Paulo (FESP)** figura como parte, com foco em:

- evolução temporal (distribuição e sentença);
- concentração geográfica (foro/vara);
- exposição financeira corrigida (IPCA);
- tempo até sentença (eficiência jurisdicional).

## 2. Amostra e dados

In [3]:
import json

import polars as pl

from config.paths import REPO_ROOT, SILVER_FACE_CLEAN, SILVER_PROCESSOS

ARTICLE_ROOT = REPO_ROOT / "projects/litigancia/notebooks/articles/eped-2026-analise-execucoes-fiscais"
ARTIFACTS_DIR = ARTICLE_ROOT / "artifacts"
FIGURES_DIR = ARTICLE_ROOT / "figures"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

PROCESSOS_COM_ERRO = [
    "1500152-68.2019.8.26.0014",
    "1503946-34.2018.8.26.0014",
    "1505517-89.2022.8.26.0114",
]

ICMS_ASSUNTOS = [
    "ICMS/ Imposto sobre Circulação de Mercadorias",
    "ICMS/Importação",
    "ICMS / Incidência Sobre o Ativo Fixo",
]

FESP_AUTOR_PATTERN = (
    r"fazenda do estado|fazenda estadual|\bfesp\b|"
    r"estado de s[aã]o paulo|faz\.?\s*estad|fda estado"
)

round2_cds = (
    pl.scan_delta(str(SILVER_PROCESSOS))
    .filter(pl.col("source_bronze_path").cast(pl.Utf8).str.contains("coleta_assunto_icms"))
    .select("cd_processo")
    .unique()
)

lf = (
    pl.scan_delta(str(SILVER_FACE_CLEAN))
    .join(round2_cds, on="cd_processo", how="inner")
    .filter(pl.col("assunto").is_in(ICMS_ASSUNTOS))
    .filter(
        pl.col("autores")
        .cast(pl.Utf8)
        .str.to_lowercase()
        .str.contains(FESP_AUTOR_PATTERN)
        .fill_null(False)
    )
    .filter(~pl.col("numero").is_in(PROCESSOS_COM_ERRO))
)

summary = lf.select(
    pl.len().alias("n_processos_fesp"),
    pl.col("distribuicao_data").min().alias("min_distribuicao"),
    pl.col("distribuicao_data").max().alias("max_distribuicao"),
    pl.col("valor_corrigido_atual").sum().alias("exposicao_ipca_total"),
    pl.col("valor_corrigido_atual").mean().alias("ticket_medio_ipca"),
    pl.col("foro").n_unique().alias("n_foros"),
    pl.col("vara").n_unique().alias("n_varas"),
).collect()

print(summary)
summary.write_csv(ARTIFACTS_DIR / "eped_2026_summary.csv")

payload = {
    "title": "ANÁLISE EMPÍRICA DAS EXECUÇÕES FISCAIS ESTADUAIS PAULISTAS ENTRE 2016 E 2025",
    "event": "XV EPED 2026",
    "submission_id": "1540866",
    "gt": "GT68",
    **{k: (str(v) if v is not None else None) for k, v in summary.row(0, named=True).items()},
}
(ARTIFACTS_DIR / "eped_2026_summary.json").write_text(
    json.dumps(payload, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)


shape: (1, 7)
┌───────────────┬───────────────┬───────────────┬───────────────┬──────────────┬─────────┬─────────┐
│ n_processos_f ┆ min_distribui ┆ max_distribui ┆ exposicao_ipc ┆ ticket_medio ┆ n_foros ┆ n_varas │
│ esp           ┆ cao           ┆ cao           ┆ a_total       ┆ _ipca        ┆ ---     ┆ ---     │
│ ---           ┆ ---           ┆ ---           ┆ ---           ┆ ---          ┆ u32     ┆ u32     │
│ u32           ┆ datetime[μs]  ┆ datetime[μs]  ┆ f64           ┆ f64          ┆         ┆         │
╞═══════════════╪═══════════════╪═══════════════╪═══════════════╪══════════════╪═════════╪═════════╡
│ 514364        ┆ 1934-06-14    ┆ 2026-07-02    ┆ 4.8710e12     ┆ 1.0188e7     ┆ 337     ┆ 61      │
│               ┆ 00:00:00      ┆ 00:00:00      ┆               ┆              ┆         ┆         │
└───────────────┴───────────────┴───────────────┴───────────────┴──────────────┴─────────┴─────────┘


418

## 3. Principais achados (slides)

Execute [`fesp_execucao_fiscal_analysis.ipynb`](fesp_execucao_fiscal_analysis.ipynb) para gerar figuras em `../figures/`.

Sugestão de ordem na apresentação oral:

1. Contexto e recorte FESP
2. Evolução longitudinal (`01_evolucao_longitudinal.png`)
3. Volume operacional entrada/saída (`02_volume_operacional.png`)
4. Concentração foro/vara — volume e valor (`03_*`, `04_*`)
5. Dispersão financeira e carga econômica (`05_*`, `06_*`)
6. Eficiência — tempo até sentença (`08_eficiencia_tempo_sentenca.png`)
7. Limitações e próximos passos

In [2]:
figures = sorted(FIGURES_DIR.glob("*.png")) if FIGURES_DIR.exists() else []
if figures:
    print("Figuras disponíveis:")
    for p in figures:
        print(f"  - {p.name}")
else:
    print("Nenhuma figura em ../figures/ — rode fesp_execucao_fiscal_analysis.ipynb primeiro.")

Figuras disponíveis:
  - 01_evolucao_longitudinal.png
  - 02_volume_operacional.png
  - 03_concentracao_foro_vara_volume.png
  - 04_concentracao_foro_vara_valor.png
  - 05_concentracao_foro_vara_valor.png
  - 06_dispersao_financeira.png
  - 07_carga_economica_ano.png
  - 08_ticket_medio_ano.png
  - 09_eficiencia_tempo_sentenca.png
  - 10_mapa_risco_foros.png
  - 11_extra_plot.png


## 4. Limitações

- `autores`/`reus` são strings do scrape, não entidades normalizadas.
- Outliers extremos de valor distorcem médias; análise usa tetos e exclusão de CNJs com erro de parsing.
- Cobertura FACE é subconjunto do acervo total de execuções fiscais TJSP.
- Dados de 2026 podem estar incompletos (coleta em andamento).

## 5. Referências no repo

- Análise completa: `fesp_execucao_fiscal_analysis.ipynb`
- Data paper: `notebooks/data_paper/`, `docs/manuscript/data_descriptor.tex`
- Codebook: `docs/data_dictionary.md`